# Reasoning Scaling Laws

Training and benchmark pipeline testing H1 (data efficiency vs. model size) and H2 (agentic compute vs. model scale) on GSM8K.

**Run order:** Cell 1 (install) → Cell 2 (download from GitHub) → Cell 3 (mount Drive) → Cell 4 (GPU check) → Cell 5 (config) → Cell 6 (train 3B) → Cell 7 (train 7B) → Cell 8 (verify) → Cell 9 (benchmark) → Cell 10 (results).

Cells 6 and 7 can be run in separate Colab sessions — checkpoints are saved to Drive after every epoch.

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:False"
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

In [2]:
# Cell 1 — Install dependencies 
!pip install -q --upgrade --no-cache-dir --force-reinstall unsloth unsloth_zoo vllm
!pip install -q datasets matplotlib wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 23.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 177.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 381.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 251.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 248.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 298.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 311.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 276.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 356.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 219.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 165.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 MB 288.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━

In [2]:
# import os; os._exit(0)
import unsloth, trl, transformers, torch, vllm
print("unsloth", unsloth.__version__)
print("trl", trl.__version__)
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("vllm", vllm.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth 2026.5.6
trl 0.24.0
transformers 4.57.6
torch 2.10.0+cu128
vllm 0.19.1


In [3]:
%%bash
# Replace your-username and your-repo-name with your actual GitHub details
BRANCH="runBranch"
USER="huineith"
REPO="AgenticAndRL_LocalMathModels"

# Define the files to download
FILES=("utils.py" "prompts.py" "rewards.py" "grpo_trainer.py" "agent.py" "benchmarker.py" "warmup_data.json")

echo "Fetching fresh files from GitHub and overwriting local copies..."
for file in "${FILES[@]}"; do
    wget -q -O "/content/$file" "https://raw.githubusercontent.com/$USER/$REPO/$BRANCH/$file"
    echo " ✓ Updated: $file"
done

Fetching fresh files from GitHub and overwriting local copies...
 ✓ Updated: utils.py
 ✓ Updated: prompts.py
 ✓ Updated: rewards.py
 ✓ Updated: grpo_trainer.py
 ✓ Updated: agent.py
 ✓ Updated: benchmarker.py
 ✓ Updated: warmup_data.json


In [4]:
# Cell 2 — Mount Google Drive (needed for checkpoint and result saving)
import os
import sys
from google.colab import drive

drive.mount('/content/drive')

CONTENT_DIR = '/content'
os.chdir(CONTENT_DIR)
if CONTENT_DIR not in sys.path:
    sys.path.insert(0, CONTENT_DIR)

print(f'Working directory: {os.getcwd()}')
print('Source files were downloaded from GitHub in the previous cell.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content
Source files were downloaded from GitHub in the previous cell.


In [5]:
# Cell 3 — GPU diagnostics
import torch

if torch.cuda.is_available():
    name  = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    free  = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
    print(f'GPU   : {name}')
    print(f'VRAM  : {total:.1f} GB total  |  {free:.1f} GB free')
    print(f'BF16  : {torch.cuda.is_bf16_supported()}')
else:
    print('WARNING: No GPU detected. Switch runtime to L4 GPU in Runtime > Change runtime type.')

GPU   : Tesla T4
VRAM  : 15.6 GB total  |  15.6 GB free
BF16  : False


In [6]:
# Cell 4 — Configuration
MODEL_3B        = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MODEL_7B        = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
DATA_FRACTIONS  = [0.10, 0.20, 0.40]
WARMUP_PATH     = 'warmup_data.json'
SEED            = 42
MAX_EPOCHS      = 3

DRIVE_SAVE_DIR  = '/content/drive/MyDrive/ExamensArbete/checkpoints'
BENCHMARK_DIR   = '/content/drive/MyDrive/ExamensArbete/benchmark_results'

print('Config:')
print(f'  3B model       : {MODEL_3B}')
print(f'  7B model       : {MODEL_7B}')
print(f'  Data fractions : {[int(f*100) for f in DATA_FRACTIONS]}%')
print(f'  Checkpoint dir : {DRIVE_SAVE_DIR}')

Config:
  3B model       : unsloth/Qwen2.5-3B-Instruct-bnb-4bit
  7B model       : unsloth/Qwen2.5-7B-Instruct-bnb-4bit
  Data fractions : [10, 20, 40]%
  Checkpoint dir : /content/drive/MyDrive/ExamensArbete/checkpoints


## Test Run (optional)

Run the cell below **instead of Cells 5–8** to do a fast end-to-end check of the whole pipeline.
It trains the 3B model on 1% of the data (≈74 examples, 1 epoch) then runs inference on 20 questions.
Total time on an L4 should be under 15 minutes.

If this cell completes without errors the full pipeline is safe to run.

In [7]:
# Test Cell — end-to-end pipeline check
# Trains 3B on 1% data then runs a 20-question mini benchmark.
# Run Cells 1-4 first so that dependencies are installed and config variables are set.

import gc
import os
import torch
from grpo_trainer import run_grpo
from benchmarker import (
    load_benchmark_questions,
    _load_trained_model,
    _run_nonagentic,
    _format_trained_prompt,
    _compute_metrics,
)
from utils import extract_tagged_answer

TEST_FRACTION    = 0.01   # ~74 training examples
TEST_CHECKPOINT  = f'{DRIVE_SAVE_DIR}/grpo_3b_1pct_best'
TEST_N_QUESTIONS = 20

# ── Step 1: Train ──────────────────────────────────────────────────────────────
print('=' * 60)
print('STEP 1: Training 3B on 1% data (1 epoch max)')
print('=' * 60)

if os.path.exists(TEST_CHECKPOINT):
    print(f'Checkpoint already exists at {TEST_CHECKPOINT} — skipping training.')
else:
    model, tokenizer, best_acc = run_grpo(
        model_name=MODEL_3B,
        data_fraction=TEST_FRACTION,
        save_dir=DRIVE_SAVE_DIR,
        warmup_path=WARMUP_PATH,
        max_epochs=1,   # single epoch keeps the test fast
        seed=SEED,
    )
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'\nTraining complete. Best val accuracy: {best_acc:.4f}')

# ── Step 2: Mini benchmark (non-agentic, 20 questions) ─────────────────────────
print()
print('=' * 60)
print(f'STEP 2: Mini benchmark ({TEST_N_QUESTIONS} questions, 3B no agent)')
print('=' * 60)

questions, solutions = load_benchmark_questions(TEST_N_QUESTIONS, seed=SEED)
model, tokenizer = _load_trained_model(TEST_CHECKPOINT)
results = _run_nonagentic(
    model, tokenizer, questions, solutions,
    _format_trained_prompt, extract_tagged_answer,
)
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

metrics = _compute_metrics(results)
print(f'\nTest results ({TEST_N_QUESTIONS} questions):')
print(f'  Accuracy   : {metrics["accuracy"]:.2%}  ({metrics["correct"]}/{metrics["total"]})')
print(f'  Avg tokens : {metrics["avg_tokens"]:.0f}')
print()
print('✓ Pipeline completed without errors.')
print('  Note: accuracy will be low at 1% data — that is expected.')
print('  A non-zero accuracy (> 0) means the reward signal is working.')

STEP 1: Training 3B on 1% data (1 epoch max)
INFO 05-23 11:43:27 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.5.6: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.19.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Standby mode is enabled. However your setting of `gpu_memory_utilization` will OOM.
Changing `gpu_memory_utilization` to 0.76.
Unsloth: WARNING - Only 0.00 GB estimated for KV cache on your 14.6 GB GPU.
This may cause an out-of-memory crash with standby mode. Consider lowering gpu_memory_utilization.
Unsloth: Your GPU cannot handle sequence lengths of 256 due to limited GPU memory.
Unsloth: Your GP

MemoryError: Unsloth: Your GPU ran out of memory loading vLLM with standby mode enabled.
Your GPU has 14.6 GB VRAM with gpu_memory_utilization=0.760.
Try one of these fixes:
  1. Lower gpu_memory_utilization: model, tokenizer = FastLanguageModel.from_pretrained(..., gpu_memory_utilization=0.6)
  2. Disable standby mode: remove os.environ['UNSLOTH_VLLM_STANDBY'] = '1'
  3. Use a smaller model or quantization (load_in_4bit=True)
Original error: CUDA out of memory. Tried to allocate 22.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 9.75 MiB is free. Process 9019 has 192.00 MiB memory in use. Process 23997 has 12.19 GiB memory in use. Process 29056 has 152.00 MiB memory in use. Including non-PyTorch memory, this process has 2.03 GiB memory in use. Of the allocated memory 1.86 GiB is allocated by PyTorch, with 40.71 MiB allocated in private pools (e.g., CUDA Graphs), and 21.79 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTOR

## Training

Cells 5 and 6 each run 3 independent GRPO training runs (one per data fraction).
Each run: SFT warmup (1 epoch) → GRPO with early stopping (max 3 epochs).
Checkpoints are saved to Drive after every epoch — safe to resume if the session disconnects.
If a `_best` checkpoint already exists for a run, that run is skipped automatically.

In [ ]:
# Cell 5 — Train 3B model across all data fractions
import gc
import os
import torch
from grpo_trainer import run_grpo

for fraction in DATA_FRACTIONS:
    pct      = int(fraction * 100)
    n_ex     = int(7500 * fraction)
    expected = f'{DRIVE_SAVE_DIR}/grpo_3b_{pct}pct_best'

    print(f'\n{"="*60}')
    print(f'3B | {pct}% data  ({n_ex} examples)')
    print(f'{"="*60}')

    if os.path.exists(expected):
        print(f'  Checkpoint already exists at {expected} -- skipping.')
        continue

    model, tokenizer, best_acc = run_grpo(
        model_name=MODEL_3B,
        data_fraction=fraction,
        save_dir=DRIVE_SAVE_DIR,
        warmup_path=WARMUP_PATH,
        max_epochs=MAX_EPOCHS,
        seed=SEED,
    )
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'  Completed. Best val accuracy: {best_acc:.4f}')

In [ ]:
# Cell 6 — Train 7B model across all data fractions
import gc
import os
import torch
from grpo_trainer import run_grpo

for fraction in DATA_FRACTIONS:
    pct      = int(fraction * 100)
    n_ex     = int(7500 * fraction)
    expected = f'{DRIVE_SAVE_DIR}/grpo_7b_{pct}pct_best'

    print(f'\n{"="*60}')
    print(f'7B | {pct}% data  ({n_ex} examples)')
    print(f'{"="*60}')

    if os.path.exists(expected):
        print(f'  Checkpoint already exists at {expected} -- skipping.')
        continue

    model, tokenizer, best_acc = run_grpo(
        model_name=MODEL_7B,
        data_fraction=fraction,
        save_dir=DRIVE_SAVE_DIR,
        warmup_path=WARMUP_PATH,
        max_epochs=MAX_EPOCHS,
        seed=SEED,
    )
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'  Completed. Best val accuracy: {best_acc:.4f}')

## Benchmark

Uses the 40% checkpoint for Groups 1-4. Group 5 is the zero-shot Qwen2.5-14B-Instruct baseline.
500 questions from the official GSM8K test split (fixed seed = 42).

In [ ]:
# Cell 7 — Verify all required checkpoints exist
import os

to_check = {
    '3B 10%': f'{DRIVE_SAVE_DIR}/grpo_3b_10pct_best',
    '3B 20%': f'{DRIVE_SAVE_DIR}/grpo_3b_20pct_best',
    '3B 40%': f'{DRIVE_SAVE_DIR}/grpo_3b_40pct_best',
    '7B 10%': f'{DRIVE_SAVE_DIR}/grpo_7b_10pct_best',
    '7B 20%': f'{DRIVE_SAVE_DIR}/grpo_7b_20pct_best',
    '7B 40%': f'{DRIVE_SAVE_DIR}/grpo_7b_40pct_best',
}

all_ok = True
for label, path in to_check.items():
    status = 'OK     ' if os.path.exists(path) else 'MISSING'
    print(f'  [{status}]  {label}  ->  {path}')
    if 'MISSING' in status:
        all_ok = False

print()
if all_ok:
    print('All checkpoints present. Ready to run benchmark.')
else:
    print('Some checkpoints are missing. Complete training before running the benchmark.')

In [ ]:
# Cell 8 — Run benchmark (all 5 groups, 500 questions)
from benchmarker import run_benchmark

CHECKPOINT_3B = f'{DRIVE_SAVE_DIR}/grpo_3b_40pct_best'
CHECKPOINT_7B = f'{DRIVE_SAVE_DIR}/grpo_7b_40pct_best'

summaries = run_benchmark(
    checkpoint_3b=CHECKPOINT_3B,
    checkpoint_7b=CHECKPOINT_7B,
    save_dir=BENCHMARK_DIR,
    n_questions=500,
    seed=SEED,
)

In [ ]:
# Cell 9 — Display results table and plot
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(f'{BENCHMARK_DIR}/benchmark_summary.csv')
df['accuracy_%']         = (df['accuracy'] * 100).round(2)
df['avg_tokens']         = df['avg_tokens'].round(1)
df['tokens_per_correct'] = df['tokens_per_correct'].round(1)

display(df[['group_name', 'accuracy_%', 'avg_tokens', 'tokens_per_correct', 'correct', 'total']])

print()
display(Image(filename=f'{BENCHMARK_DIR}/benchmark_plot.png'))